##### The Five Principles
**Atomic Operations**: Each tool should do one thing well. A tool called manage_users that can create, read, update, delete, and send emails to users is doing too much. Break it into create_user, get_user, update_user, delete_user, and send_user_email.

**Clear Naming**: Tool names should be self-documenting verb-noun pairs. Claude reads tool names to decide which tool to use -- a clear name guides it toward the right choice.

**Exhaustive Descriptions**: The tool description is the single most important field in your tool definition. The description IS the documentation for Claude. It reads this text to decide when and how to use the tool.

**Typed Parameters**: Use JSON Schema types, enums, and required fields to constrain what Claude sends. Typed parameters prevent entire classes of errors before they happen.

**Error Clarity**: Return structured errors, not generic "something went wrong" messages. When Claude receives a clear error, it can decide whether to retry, adjust parameters, or inform the user. A vague error leaves Claude guessing.



In [ ]:
{
  "name": "search_contacts",
  "description": "Search the CRM for contacts matching the given criteria. Returns up to 'limit' results sorted by the specified field. Use this tool when the user asks about customers, contacts, or account holders. For company-level queries, use search_companies instead.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Free-text search query. Searches across name, email, and notes fields."
      },
      "status": {
        "type": "string",
        "enum": ["active", "inactive", "lead", "churned"],
        "description": "Filter by contact status. Omit to include all statuses."
      },
      "limit": {
        "type": "integer",
        "description": "Maximum number of results to return. Default is 10, max is 100.",
        "default": 10,
        "minimum": 1,
        "maximum": 100
      },
      "sort_by": {
        "type": "string",
        "enum": ["name", "created_at", "last_activity"],
        "description": "Field to sort results by. Defaults to relevance when a query is provided, or 'created_at' otherwise."
      }
    },
    "required": ["query"],
    "additionalProperties": false
  }
}

##### MCP Servers
###### What MCP serves
**Tools**: Functions that Claude can call to perform actions. These are the most common capability. Examples: querying a database, sending an email, creating a file, calling an API.

**Resources**: Data that Claude can browse without making function calls. Resources are like a catalog or directory. Examples: a list of database tables, file listings, issue trackers, documentation indexes. Claude reads these to understand what data is available.

**Prompts**: Pre-built prompt templates that guide Claude's behavior for specific workflows. Examples: a "code review" prompt template, a "data analysis" template, a "bug report" template. These help standardize how Claude approaches recurring tasks.

###### Examples of MCP Servers
DB:
{ "command": "python3", "args": ["db_mcp.py"], "env": {"DB_URL": "${DATABASE_URL}"} }
API_GATEWAY:
{ "url": "https://mcp.company.com/api-gateway", "headers": {"Authorization": "Bearer ${API_TOKEN}"} }
FILE_SYSTEM:
{ "command": "npx", "args": ["@mcp/filesystem", "/project/path"] }
SEARCH_ENGINE:
{ "url": "https://mcp.company.com/search", "headers": {"X-API-Key": "${SEARCH_KEY}"} }


In [ ]:
# Building MCP Servers - Python with FastMCP

from fastmcp import FastMCP
import json

# Create the MCP server
mcp = FastMCP("my-tools")

@mcp.tool()
def search_database(query: str, limit: int = 10) -> str:
    """Search the product database.

    Args:
        query: Search keywords to match against product
               names and descriptions.
        limit: Maximum number of results to return
               (default 10, max 100).
    """
    results = db.search(query, limit)
    return json.dumps(results)

@mcp.tool()
def get_product_by_id(product_id: str) -> str:
    """Get detailed product information by its unique ID.

    Args:
        product_id: The unique product identifier
                    (e.g. 'PROD-12345').
    """
    product = db.get(product_id)
    if not product:
        return json.dumps({"error": "Product not found"})
    return json.dumps(product)

# Run the server (uses stdio transport by default)
mcp.run()

In [ ]:
# Exposing Resources
@mcp.resource("schema://tables") def get_database_schema() -> str: """List all database tables and their columns.""" tables = db.get_schema() return json.dumps(tables, indent=2)

In [ ]:
# Testing Your MCP Server Locally
# Install the MCP inspector
npx @modelcontextprotocol/inspector python3 my_server.py

# This opens a web UI where you can:
# - See all tools your server exposes
# - Test tool calls with sample inputs
# - Inspect resources and prompts
# - Debug connection issues